In [1]:
import pandas as pd
demo_df = pd.read_parquet(r"D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\FastAPI\app\data\demo_lookup.parquet")
print(f"Total ID valid: {len(demo_df)}")
print(demo_df["SK_ID_CURR"].sample(20).tolist())  # ambil 20 ID acak sebagai contoh tambahan

Total ID valid: 61503
[413916, 125342, 320872, 165508, 186711, 315370, 455040, 406760, 263529, 280199, 426836, 211281, 218035, 337622, 363485, 299095, 230700, 450024, 343526, 243497]


In [1]:
import sys, json
import pandas as pd
import joblib

sys.path.insert(0, "src")  # agar class CatBoostColumnSelector ketemu
pipeline = joblib.load("models/catboost_v3/catboost_v3_pipeline.joblib")

with open("data/processed/feature_engineering_metadata_v3.json") as f:
    metadata = json.load(f)
tree_features = metadata["tree_features_v3"]

demo_df = pd.read_parquet("FastAPI/app/data/demo_lookup.parquet")
X = demo_df[tree_features]
demo_df["proba"] = pipeline.predict_proba(X)[:, 1]

# Ambil ID mendekati beberapa titik probabilitas kunci
for target_p in [0.05, 0.30, 0.60, 0.6669, 0.72, 0.90]:
    row = demo_df.iloc[(demo_df["proba"] - target_p).abs().argsort()[:1]]
    print(f"target={target_p:.4f} -> SK_ID_CURR={row['SK_ID_CURR'].values[0]}, proba={row['proba'].values[0]:.4f}")

# Cari ID dengan EXT_SOURCE hilang
missing_ext = demo_df[demo_df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].isna().any(axis=1)]
print("\nContoh ID dengan EXT_SOURCE hilang:")
print(missing_ext[["SK_ID_CURR", "proba"]].head(5))

target=0.0500 -> SK_ID_CURR=426033, proba=0.0499
target=0.3000 -> SK_ID_CURR=359624, proba=0.3000
target=0.6000 -> SK_ID_CURR=272824, proba=0.6000
target=0.6669 -> SK_ID_CURR=114228, proba=0.6669
target=0.7200 -> SK_ID_CURR=255863, proba=0.7200
target=0.9000 -> SK_ID_CURR=366811, proba=0.9001

Contoh ID dengan EXT_SOURCE hilang:
Empty DataFrame
Columns: [SK_ID_CURR, proba]
Index: []
